In [2]:
# Import Python Standard Library dependencies
from copy import copy
import datetime
from glob import glob
import json
import math
import multiprocessing
import os
from pathlib import Path
import random
import urllib.request

# Import utility functions
from cjm_pandas_utils.core import markdown_to_pandas
from cjm_pil_utils.core import resize_img, get_img_files
from cjm_psl_utils.core import download_file, file_extract
from cjm_pytorch_utils.core import set_seed, pil_to_tensor, tensor_to_pil, get_torch_device, denorm_img_tensor
from cjm_torchvision_tfms.core import ResizeMax, PadSquare

import matplotlib.pyplot as plt 
import numpy as np
import pandas as pd

# Do not truncate the contents of cells and display all rows and columns
pd.set_option('max_colwidth', None, 'display.max_rows', None, 'display.max_columns', None)

# Import PIL for image manipulation
from PIL import Image

# Import timm library
import timm

# Import PyTorch dependencies
import torch
import torch.nn as nn
from torch.amp import autocast
from torch.cuda.amp import GradScaler
from torch.utils.data import Dataset, DataLoader

import torchvision
torchvision.disable_beta_transforms_warning()
import torchvision.transforms.v2  as transforms
from torchvision.transforms.v2 import functional as TF

from torchtnt.utils import get_module_summary
from torcheval.metrics import MulticlassAccuracy

# Import tqdm for progress bar
from tqdm.auto import tqdm

In [3]:
#Set the seed for generating random numbers in PyTorch, NumPy, and Python's random module
seed = 1234
set_seed(seed)

In [3]:
device = get_torch_device()
dtype = torch.float32
device, dtype

('cuda', torch.float32)

In [4]:
#The path for the project folder
project_dir = Path(r"C:\Users\harri\Documents\Image-Classifier")

#Create the project directory if it does not already exist
project_dir.mkdir(parents=True, exist_ok=True)

#Define path to store datasets
dataset_dir = project_dir/'data'
#Create the dataset directory if it does not exist
dataset_dir.mkdir(parents=True, exist_ok=True)

#Define path to store archive files
archive_dir = dataset_dir/'Archive'
#Create the archive directory if it does not exist
archive_dir.mkdir(parents=True, exist_ok=True)

#Creating a Series with the paths and converting it to a DataFrame for display
pd.Series({
    "Project Directory:": project_dir,
    "Dataset Directory:": dataset_dir,
    "Archive Directory:": archive_dir
}).to_frame().style.hide(axis='columns')

Project Directory:,C:\Users\harri\Documents\Image-Classifier
Dataset Directory:,C:\Users\harri\Documents\Image-Classifier\data
Archive Directory:,C:\Users\harri\Documents\Image-Classifier\data\Archive


In [5]:
#Set the name of the dataset
dataset_name = 'hagrid-classification-512p-no-gesture-150k-zip'

#Construct the HuggingFace Hub dataset name by combining the username and dataset name
hf_dataset = f'cj-mills/{dataset_name}'

#Create the path to the zip file that contains the dataset
archive_path = Path(f'{archive_dir}/{dataset_name.removesuffix("-zip")}.zip')

#Create the path to the directory where the dataset will be extracted
dataset_path = Path(f'{dataset_dir}/{dataset_name.removesuffix("-zip")}')

#Creating a Series with the dataset name and paths and converting it to a DataFrame for display
pd.Series({
    "HuggingFace Dataset:": hf_dataset,
    "Archive Path:": archive_path,
    "Dataset Path": dataset_path
}).to_frame().style.hide(axis='columns')

HuggingFace Dataset:,cj-mills/hagrid-classification-512p-no-gesture-150k-zip
Archive Path:,C:\Users\harri\Documents\Image-Classifier\data\Archive\hagrid-classification-512p-no-gesture-150k.zip
Dataset Path,C:\Users\harri\Documents\Image-Classifier\data\hagrid-classification-512p-no-gesture-150k


In [6]:
#Construct the HuggingFace Hub dataset URL
dataset_url = f"https://huggingface.co/datasets/{hf_dataset}/resolve/main/{dataset_name.removesuffix('-zip')}.zip"
print(f"HuggingFace Dataset URL: {dataset_url}")
#Set whether to delete archive file after extracting the dataset
delete_archive = True

#Download the dataset if not present
if dataset_path.is_dir():
    print("Dataset folder already exists")
else:
    print("Downloading dataset...")
    download_file(dataset_url, archive_dir)

    print("Extracting dataset...")
    file_extract(fname=archive_path, dest=dataset_dir)

    #Delete the archive if specified
    if delete_archive: archive_path.unlink()

HuggingFace Dataset URL: https://huggingface.co/datasets/cj-mills/hagrid-classification-512p-no-gesture-150k-zip/resolve/main/hagrid-classification-512p-no-gesture-150k.zip
Dataset folder already exists


In [7]:
img_folder_paths = [folder for folder in dataset_path.iterdir() if folder.is_dir()]
#Display the names of the folders using a Pandas DataFrame
pd.DataFrame({"Image Folder": [folder.name for folder in img_folder_paths]})

,Image Folder
0,call
1,dislike
2,fist
3,four
4,like
5,mute
6,no_gesture
7,ok
8,one
9,palm


In [8]:
#Get a list of all image file paths from the imge folders
class_file_paths = [get_img_files(folder) for folder in img_folder_paths]

#Get all image files in the 'img_dir' directory
img_paths = [
    file
    for folder in class_file_paths #Iterate through each image folder
    for file in folder #Get a list of image files in each image folder
]

#Print the number of image files
print(f"Number of Images: {len(img_paths)}")

#Display the first five entries using a Pandas DataFrame
pd.DataFrame(img_paths).head()


Number of Images: 143736


,0
0,C:\Users\harri\Documents\Image-Classifier\data\hagrid-classification-512p-no-gesture-150k\dislike\6f246425-9448-46b3-bc3b-dfa6f6eb302d.jpeg
1,C:\Users\harri\Documents\Image-Classifier\data\hagrid-classification-512p-no-gesture-150k\dislike\6f2dc79b-3bcd-42aa-9087-5de0922081af.jpeg
2,C:\Users\harri\Documents\Image-Classifier\data\hagrid-classification-512p-no-gesture-150k\dislike\6f3ffd85-0ffb-46ab-9a8d-9a0a3e30fd47.jpeg
3,C:\Users\harri\Documents\Image-Classifier\data\hagrid-classification-512p-no-gesture-150k\dislike\6f4c0855-3a02-4abc-b6e9-f05754d785fc.jpeg
4,C:\Users\harri\Documents\Image-Classifier\data\hagrid-classification-512p-no-gesture-150k\dislike\6f4f7cb1-82c5-419b-a633-82a6c6d2b37f.jpeg


In [9]:
#Get the number of samples for each image class
class_counts_dict = {folder[0].parent.name:len(folder) for folder in class_file_paths}

#Get a list of unique labels
class_names = list(class_counts_dict.keys())

#Display the labels and the corresponding number of samples using a Pandas DataFrame
class_counts = pd.DataFrame.from_dict({'Count': class_counts_dict})

IndexError: list index out of range